In [14]:
from pathlib import Path
from pptx import Presentation
from docx import Document
from unstructured.partition.auto import partition
from langchain.text_splitter import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
import requests
# Add these to existing imports
from datetime import datetime

hist_texts, hist_metas = [], []

In [2]:
from docx import Document  # ADD THIS at the top

def extract_text_from_pptx(file_path):
    try:
        prs = Presentation(file_path)
        text = [shape.text for slide in prs.slides for shape in slide.shapes if hasattr(shape, "text")]
        return "\n".join(text)
    except Exception:
        return None

def extract_text_from_docx(file_path):
    try:
        doc = Document(file_path)
        return "\n".join([para.text for para in doc.paragraphs])
    except Exception:
        return None

def extract_text_with_unstructured(file_path):
    try:
        elements = partition(file_path=file_path)
        return "\n".join([str(el) for el in elements])
    except Exception:
        return None

def extract_text(file_path):
    ext = Path(file_path).suffix.lower()
    if ext == ".pptx":
        text = extract_text_from_pptx(file_path)
    elif ext == ".docx":
        text = extract_text_from_docx(file_path)
    elif ext == ".pdf":
        text = extract_text_with_unstructured(file_path)
    else:
        text = None

    return text or extract_text_with_unstructured(file_path) or ""


In [3]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)

base_path = Path("Diss_doc")  # 🔁 replace with actual path

all_chunks = []

for folder in base_path.iterdir():
    if folder.is_dir():
        for file in folder.rglob("*"):
            if file.suffix.lower() in [".pptx", ".docx", ".pdf"]:
                raw_text = extract_text(file)
                if raw_text:
                    chunks = text_splitter.split_text(raw_text)
                    for chunk in chunks:
                        all_chunks.append({
                            "content": chunk,
                            "source": f"{folder.name}/{file.name}"
                        })


In [4]:
model = SentenceTransformer("all-MiniLM-L6-v2")
texts = [c["content"] for c in all_chunks]
metas = [c["source"] for c in all_chunks]

embeddings = model.encode(texts, show_progress_bar=True, normalize_embeddings=True)

index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(np.array(embeddings))


Batches: 100%|██████████| 82/82 [00:02<00:00, 28.83it/s]


In [5]:
def search(query, top_k=5):
    query_vec = model.encode([query], normalize_embeddings=True)
    scores, indices = index.search(np.array(query_vec), top_k)
    
    for i, idx in enumerate(indices[0]):
        print(f"\n--- Match {i+1} ---")
        print(f"Cosine Similarity: {scores[0][i]:.4f}")
        print(f"Source: {metas[idx]}")
        print(f"Content:\n{texts[idx][:500]}...")


In [6]:
search("project discovery phase for a fintech client")



--- Match 1 ---
Cosine Similarity: 0.5954
Source: Proposals/Copy of Informa Procurement Technology Cost Optimisation - Savings Discovery v2.pptx
Content:
Key: Scope of discovery work
Proposed team & commercials 
[ ‹#› ]
We are proposing the following team structure for this initial phase, to mobilise the programme, demonstrate a repeatable approach that can be both scaled and utilised in stakeholder engagement. 
Delivery of this initial phase is estimated at £57k (excluding VAT). Our day rates are based on the current Informa-Clarasys rate card....

--- Match 2 ---
Cosine Similarity: 0.5523
Source: Proposals/Copy of Informa Procurement Technology Cost Optimisation - Savings Discovery v2.pptx
Content:
Our team will work with Procurement, Technology & Business Owners on the Discovery. Additionally, we will work in close collaboration with the wider programme team; focusing on the scope of WP2 over a seven week discovery period (start date to be confirmed). We will also provide supportin

In [7]:
def query_mistral(prompt, model="mistral:latest"):
    response = requests.post(
        "http://localhost:11434/api/generate",
        json={
            "model": model,
            "prompt": prompt,
            "stream": False
        }
    )
    return response.json()['response']


In [8]:
chat_history = []

def get_expanded_query(user_input, chat_history, n_prev=1):
    history = [
        msg["content"]
        for msg in chat_history[-2 * n_prev : -1]
        if msg["role"] in ("user", "assistant")
    ]
    return "\n".join(history + [user_input])


In [9]:
def build_prompt(query, context_chunks_with_sources, history_chunks=None):
    history_text = ""
    if history_chunks:
        history_text = "\n<history>\n".join(
            f"[{src}]\n{chunk}" for chunk, src in history_chunks
        ) + "\n</history>"

    docs_text = "\n<doc>\n".join(
        f"[{src}]\n{chunk}" for chunk, src in context_chunks_with_sources
    ) + "\n</doc>"

    prompt = f"""You are a senior consultant leading the discovery phase of a client project.

You are given excerpts from past project documents and relevant previous conversation history. 
These are strictly your ONLY source of information. Do NOT use any external knowledge.

--- START OF HISTORY ---
{history_text if history_text else "No previous history used"}
--- END OF HISTORY ---

--- START OF CONTEXT ---
{docs_text}
--- END OF CONTEXT ---

Client request:
"{query}"

Your task:

- List all clear, stated client requirements from the request (not inferred ones).
- Find similar past projects based only on visible matches in the excerpts and/or relevant history.
- For each similar project, include:
  - The [filename] or [History_x]
  - A short fact-based summary of what was done
- Mention tools, industries, or outcomes only if they appear directly in the context or history.
- Output strictly in this bullet list format:
  - [filename/history_id]: short factual statement
  - technologies used
  - industries involved
  - a basic timeframe
"""
    return prompt


In [15]:
def rag(query, top_k=10, min_score=0.5, max_context_chunks=8, top_k_hist=3):
    query_vec = model.encode([query], normalize_embeddings=True)

    # Document retrieval
    scores, indices = index.search(np.array(query_vec), top_k)
    context_chunks_with_sources, seen_sources = [], set()
    for i, idx in enumerate(indices[0]):
        if scores[0][i] >= min_score and metas[idx] not in seen_sources:
            context_chunks_with_sources.append((texts[idx], metas[idx]))
            seen_sources.add(metas[idx])
            if len(context_chunks_with_sources) >= max_context_chunks:
                break

    # History retrieval
    # History retrieval (only if history is initialized and non-empty)
    history_chunks = []
    if 'hist_index' in globals() and hist_texts:
        hist_scores, hist_indices = hist_index.search(np.array(query_vec), top_k_hist)
        for i, h_idx in enumerate(hist_indices[0]):
            if hist_scores[0][i] >= min_score:
                history_chunks.append((hist_texts[h_idx], f"History_{h_idx}"))

    if not context_chunks_with_sources and not history_chunks:
        return "Not enough info."

    prompt = build_prompt(query, context_chunks_with_sources, history_chunks)
    response = query_mistral(prompt)

    add_history_entry(query, response, meta={"type": "rag_turn"})
    return response


In [11]:
q1="""From:
Anita Desai
Head of Digital Services
City of Riverton Council

Subject:
Request for Support on Citizen Services Modernization Initiative

Hi team,

We’re reaching out to explore working with your consultancy on a new initiative we're planning.

The City of Riverton is looking to modernize how we deliver public services online. Our current systems are fragmented and heavily manual — for example, applying for permits, reporting local issues, and accessing social support all happen on different platforms (some even still on paper).

Our aim is to create a unified digital experience for citizens. This would include:

    A central online portal where people can log in and access multiple services

    Backend automation to reduce manual workloads for our staff

    Better tracking and analytics to understand service usage and pain points

We’re particularly keen to learn what similar projects you’ve worked on with other public sector bodies, and how we can avoid common pitfalls. We know this will be a multi-phase project, and right now we’re just trying to get a clear picture of what the discovery and planning would look like.

Looking forward to hearing how you’d approach this.

Thanks,
Anita
    """


In [16]:
# 🔁 Chat loop
history = []

print("Chat started. Type 'exit' to stop.\n")

while True:
    user_input = input("You: ")
    if user_input.lower() == "exit":
        break

    # response = rag(user_input, history)
    response = rag(user_input)  
    print(user_input)
    print("\nConsultant:\n", response)

    history.append({"role": "user", "content": user_input})
    history.append({"role": "assistant", "content": response})


Chat started. Type 'exit' to stop.



NameError: name 'add_history_entry' is not defined